In [1]:
! pip install requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [requests]


In [3]:
COURTLISTENER_API_TOKEN = "2a376d42b598b6a485df429843829e5ed134e676"

In [7]:
import os
import requests

# Set this once in your shell: export COURTLISTENER_API_TOKEN=...
API_TOKEN = COURTLISTENER_API_TOKEN
if not API_TOKEN:
    raise ValueError("Missing COURTLISTENER_API_TOKEN environment variable")

url = "https://www.courtlistener.com/api/rest/v4/search/"

params = {
    "q": "theft in the city",
    "type": "o",  # opinions
    "page_size": 5,
}

headers = {
    "Authorization": f"Token {API_TOKEN}",
    "Accept": "application/json",
}

response = requests.get(url, params=params, headers=headers, timeout=20)
response.raise_for_status()
data = response.json()

evidence_snippets = []
for rank, result in enumerate(data.get("results", []), start=1):
    case_name = result.get("caseName") or result.get("caseNameShort") or "Unknown case"
    absolute_url = result.get("absolute_url")
    if absolute_url and absolute_url.startswith("/"):
        absolute_url = f"https://www.courtlistener.com{absolute_url}"

    snippet = result.get("snippet") or result.get("text") or case_name
    evidence_snippets.append(
        {
            "source_name": "courtlistener",
            "citation": absolute_url or "",
            "snippet": snippet,
            "relevance": max(0.0, 1.0 - (rank - 1) * 0.1),
        }
    )

for item in evidence_snippets:
    print(item["citation"] or "(no citation)")
    print(item["snippet"][:220])
    print(f"Relevance: {item['relevance']:.2f}")
    print("-" * 50)

https://www.courtlistener.com/opinion/8769042/in-re-grand-theft-auto-video-game-consumer-litigation/
In re Grand Theft Auto Video Game Consumer Litigation
Relevance: 1.00
--------------------------------------------------
https://www.courtlistener.com/opinion/2477568/in-re-grand-theft-auto-video-game-consumer-litigation/
In Re Grand Theft Auto Video Game Consumer Litigation
Relevance: 0.90
--------------------------------------------------
https://www.courtlistener.com/opinion/5640728/in-re-daniel-h/
In re Daniel H.
Relevance: 0.80
--------------------------------------------------
https://www.courtlistener.com/opinion/2183709/holocaust-victims-of-bank-theft-v-magyar-nemzeti-bank/
Holocaust Victims of Bank Theft v. Magyar Nemzeti Bank
Relevance: 0.70
--------------------------------------------------
https://www.courtlistener.com/opinion/2113884/holocaust-victims-of-bank-theft-v-magyar-nemzeti-bank/
Holocaust Victims of Bank Theft v. Magyar Nemzeti Bank
Relevance: 0.60
----------------